# 2D-to-3DGS — Colab orchestrator

Training code stays in `.py` modules. This notebook mounts Drive, installs deps, stages data to fast local disk (`/content/data`), and runs `train.py` via CLI.

**Setup:** `setup.sh` skips `mamba-ssm` by default (Colab often fails building those CUDA wheels). The ViM encoder uses a built-in mixer fallback. To try real Mamba: `INSTALL_MAMBA=1 bash setup.sh` (may still fail).

**Dataset zip (optional):** Set **one** of `dataset_gdrive_id`, `dataset_download_url`, `dataset_drive_zip`, or `dataset_archive` in `configs/colab_config.yaml`, then run the **Zip download / extract** cell.

**Training data:** `train.py` loads **only real images** from `data_root/manifest.jsonl` (no synthetic fallback). The repo ships a tiny **`data/demo`** bundle; default `colab_config.yaml` points `data_root` there so a fresh clone runs. For scale, unzip your image zip (next cells) or build a full `manifest.jsonl` under e.g. `/content/data/...` and set `data_root`.

**Objaverse vehicle GLBs:** Optional download when `objaverse_download: true`. GLBs still need rendering + `manifest.jsonl` before they become training data. License: [allenai/objaverse](https://huggingface.co/datasets/allenai/objaverse).

**Imports:** Run the clone/`%cd` cell before any cell that imports `utils` or runs `train.py`. The download and train cells force `PROJECT_ROOT` on `sys.path` so rerunning a single cell still works if the repo is at `/content/2d-to-3d`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Clone on first run; on later runs `git pull` picks up new files (e.g. utils/dataset_download.py).
!if [ ! -d /content/2d-to-3d ]; then git clone https://github.com/ns-1456/2d-to-3d.git /content/2d-to-3d; else git -C /content/2d-to-3d pull; fi
%cd /content/2d-to-3d

In [ ]:
# Default install (no mamba-ssm CUDA build). For optional Mamba: INSTALL_MAMBA=1 bash setup.sh
!chmod +x setup.sh && bash setup.sh

In [ ]:
# Download Objaverse vehicle GLBs (see configs/colab_config.yaml: objaverse_*).
# Requires clone + cd cell above. Uses open-license data; compliance is your responsibility.
import os
import subprocess
import sys

REPO_DIR = "/content/2d-to-3d"
if not os.path.isdir(os.path.join(REPO_DIR, "utils")):
    raise FileNotFoundError(f"Run the git clone + %%cd cell first. Missing {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements_data.txt"],
    check=True,
)

import yaml

with open("configs/colab_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

if not cfg.get("objaverse_download", False):
    print("Skipping Objaverse download (objaverse_download: false in colab_config.yaml).")
else:
    out_dir = cfg.get("objaverse_out_dir", "/content/data/objaverse_vehicles")
    cache = cfg.get("objaverse_cache", "/content/data/.objaverse_cache")
    mx = int(cfg.get("objaverse_max_objects", 100))
    mode = str(cfg.get("objaverse_mode", "both"))
    scan = int(cfg.get("objaverse_scan_uids", 100_000))
    procs = int(cfg.get("objaverse_processes", 4))
    cmd = [
        sys.executable,
        "scripts/download_objaverse_vehicles.py",
        "--out_dir",
        out_dir,
        "--max_objects",
        str(mx),
        "--mode",
        mode,
        "--scan_uids",
        str(scan),
        "--objaverse_cache",
        cache,
        "--download_processes",
        str(procs),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Done. GLBs under:", out_dir, "| Render to PNGs + add manifest.jsonl under data_root before training on them.")

In [ ]:
# Optional: staged .zip of IMAGES + manifest (not Objaverse GLBs). See colab_config.yaml:
#   dataset_gdrive_id, dataset_download_url, dataset_drive_zip, or dataset_archive
!pip install -q gdown

import os
import sys

REPO_DIR = "/content/2d-to-3d"
if not os.path.isdir(os.path.join(REPO_DIR, "utils")):
    raise FileNotFoundError(
        f"Missing {REPO_DIR}/utils — run the clone + %%cd cell above first."
    )
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import yaml

from utils.colab_setup import unzip_to_local
from utils.dataset_download import resolve_local_zip_path

with open("configs/colab_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

root = cfg["data_root"]
zip_path = resolve_local_zip_path(cfg)
if zip_path:
    print("Local zip:", zip_path)
    unzip_to_local(zip_path, root, overwrite=False)
    print("Extracted to", root, "- ensure manifest.jsonl + images exist; set data_root in YAML to this path.")
else:
    print(
        "No image zip configured. If data_root already has manifest.jsonl (e.g. bundled data/demo), training is fine. "
        "Otherwise set dataset_* in colab_config.yaml and re-run this cell."
    )

In [ ]:
# Must run from repo root so `data` / `models` package imports resolve.
%cd /content/2d-to-3d
!python train.py --config configs/colab_config.yaml
# Resume after disconnect:
# !python train.py --config configs/colab_config.yaml --resume_from /content/drive/MyDrive/3DGS_Checkpoints